# 中际旭创（300308）做T横线指标分钟级回测

本笔记使用本地逐笔成交重建分钟线。信号在分钟收盘后确认，卖出按下一分钟第一笔成交价，以避免同一分钟偷看未来。每个交易日只执行第一次确认信号。

In [1]:
from pathlib import Path
import pandas as pd

root = Path.cwd()
out = root / 'backtest_outputs'
summary = pd.read_csv(out / 'summary.csv')
trades = pd.read_csv(out / 'trades.csv')
qa = pd.read_csv(out / 'data_quality.csv')
sweep = pd.read_csv(out / 'threshold_sweep.csv')
print(f'样本：{len(qa)}个交易日，{qa.date.min()} 至 {qa.date.max()}')
print(f'日线收盘与逐笔末价差异>0.1%的天数：{(qa.close_abs_diff_pct > .001).sum()}')


样本：308个交易日，2025-04-25 至 2026-07-31
日线收盘与逐笔末价差异>0.1%的天数：0


In [2]:
cols = ['variant','period','signal_days','same_day_buyback_rate','by_next_day_buyback_rate','fly_rate']
view = summary[cols].copy()
for c in cols[3:]: view[c] = (view[c] * 100).round(1).astype(str) + '%'
print(view.to_string(index=False))


     variant        period  signal_days same_day_buyback_rate by_next_day_buyback_rate fly_rate
       fixed          full          180                 72.8%                    80.0%    20.0%
       fixed earlier_70pct          125                 76.0%                    81.6%    18.4%
       fixed  recent_30pct           55                 65.5%                    76.4%    23.6%
vwap_dynamic          full          161                 69.6%                    79.5%    20.5%
vwap_dynamic earlier_70pct          107                 72.0%                    80.4%    19.6%
vwap_dynamic  recent_30pct           54                 64.8%                    77.8%    22.2%


## 交易定义

接回成功：卖出后至下一交易日最低成交价至少低于卖价0.8%。抛飞：期间没有0.8%接回机会，且价格又上涨至少2%。0.8%为毛差价，未扣个别券商佣金。

In [3]:
fixed = trades[trades.variant.eq('fixed')].copy()
minute_no = fixed.signal_time.str[:2].astype(int)*60 + fixed.signal_time.str[3:5].astype(int)
fixed['time_bucket'] = pd.cut(minute_no,[569,600,690,779,840,901],labels=['09:30-10:00','10:01-11:30','午间','13:00-14:00','14:01-15:00'])
by_time = fixed.groupby('time_bucket', observed=True).agg(signals=('date','size'),same_day=('buyback_same_0_8','mean'),by_next=('buyback_by_next_0_8','mean'),fly=('fly_2pct_without_buyback','mean'))
print((by_time.assign(same_day=lambda x:(x.same_day*100).round(1),by_next=lambda x:(x.by_next*100).round(1),fly=lambda x:(x.fly*100).round(1))).to_string())


             signals  same_day  by_next   fly
time_bucket                                  
09:30-10:00      127      79.5     83.5  16.5
10:01-11:30       32      68.8     75.0  25.0
13:00-14:00       16      50.0     75.0  25.0
14:01-15:00        5       0.0     40.0  60.0


In [4]:
recent = sweep[sweep.period.eq('recent_30pct')][['lower_pct','signal_days','same_day_buyback_rate','by_next_day_buyback_rate','fly_rate']].copy()
recent['lower_pct'] = (recent.lower_pct*100).round(1)
for c in ['same_day_buyback_rate','by_next_day_buyback_rate','fly_rate']: recent[c] = (recent[c]*100).round(1)
print(recent.to_string(index=False))


 lower_pct  signal_days  same_day_buyback_rate  by_next_day_buyback_rate  fly_rate
       1.0           66                   75.8                      80.3      19.7
       1.2           64                   73.4                      78.1      21.9
       1.4           61                   68.9                      78.7      21.3
       1.6           59                   64.4                      76.3      23.7
       1.8           55                   65.5                      76.4      23.6
       2.0           51                   66.7                      78.4      21.6
       2.2           49                   73.5                      87.8      12.2
       2.4           48                   75.0                      87.5      12.5
       2.6           48                   77.1                      89.6      10.4
       2.8           47                   76.6                      87.2      12.8
       3.0           44                   79.5                      86.4      13.6


## 结论

固定1.8%下沿的全样本次日内0.8%接回率为80.0%，最近30%样本降至76.4%，不能称为高确定性。分时均价动态下沿没有稳定提高结果。信号集中在10:00前时表现更好，下午尤其14:00后的样本少且抛飞明显更高。大盘/通信板块历史分钟数据不在本地样本中，因此没有把收盘后数据冒充盘中条件。